# Computer Vision with SUBSIM on EDITO - Playday Quickstart

Welcome! Today you'll train an object-detection model to recognise fish species in underwater footage from Swedish coastal waters - including the **invasive round goby** (*Neogobius melanostomus*).

What you'll do:
1) Download a small YOLO-formatted goby dataset (~100 labelled underwater images)
2) Train a small YOLO model on the dataset (~15–20 epochs)
3) Inspect training outputs - loss curves, metrics, evaluation plots
4) Run inference on held-out images of alien species

**Time:** ~60 minutes total. Each step includes a short walkthrough, then time for you to run it yourself.

## Step 1 - Download the playday data

Run the cell below to download and unzip the playday dataset.

In [ ]:
!./download_playday_data.sh

Once downloaded, look in your file browser on the left. You should see the following directories:

```raw
01_dataset_goby100/       ← Your training dataset
├── data.yaml                  ← Dataset config (class names, paths)
├── train/                     ← 70 images + labels (model learns from these)
│   ├── images/
│   └── labels/
├── valid/                     ← 20 images + labels (monitored during training)
│   ├── images/
│   └── labels/
└── test/                      ← 10 images + labels (final evaluation, never seen during training)
    ├── images/
    └── labels/
02_inference/           ← Held-out images for running inference
├── goby_images/
│   ├── heldout_01.jpg
│   └── heldout_02.jpg
└── video_inference/                    ← (optional) short video clip
    └── golden_weights_clip.mp4
03_golden_model/              ← Pre-trained model + eval artifacts
├── weights/                  ← Model weights (YOLO-style)
│   └── best.pt
├── test_eval/                ← Selected test evaluation outputs
│   ├── confusion_matrix.png
│   ├── results.csv
│   ├── labels.jpg
│   └── val_batch0_pred.jpg   ← plus a few illustrative examples
└── train_plots/              ← Training curves only
    ├── BoxF1_curve.png
    ├── BoxP_curve.png
    ├── BoxR_curve.png
    └── results.png
```

**Take a moment to explore:**
- Open `01_dataset_goby100/` — click on a few images and their matching label files
- Each `.txt` label file contains one line per detected object: `class_id, x_center, y_center, width, height`
- The `data.yaml` file lists the species classes the model will learn to detect

**The species in this dataset:**

| Class | Species | Notes |
|:------|:--------|:------|
| 0 | *Ctenolabrus rupestris* (goldsinny wrasse) | Native |
| 1 | *Gobius niger* (black goby) | Native |
| 2 | **Neogobius melanostomus** (round goby) | **Invasive** |

The round goby is one of Europe's most invasive and damaging fish species. It competes with native gobies for food and habitat, and is known to destroy mussel beds and consume the roe of other fish. Early detection matters, and that's exactly what your model will learn to do.


## Step 2 - Train a YOLO model

Open the notebook: [**Train_models_playday.ipynb**](Train_models_playday.ipynb).

This notebook trains a YOLO object-detection model on your dataset. You'll run it cell by cell.

**What you'll edit in Phase 1 (Configuration):**

- `data_path` -> point to `01_dataset_goby100`
- `exp_name` -> give your experiment a name (e.g. `"my_first_model"`)
- `baseline_weights` -> set to `"yolo11n.pt"` (nano model, good for ~100 images)
- `epochs` -> start with `20` (takes a few minutes). More epochs = more learning time, but diminishing returns

**Then run all cells top to bottom.** Here's what happens at each phase:

- **Phase 2** checks your dataset and GPU are ready
- **Phase 3** runs the actual training -> watch the table update each epoch. The `mAP50` column is your main quality score (higher = better)
- **Phase 4** evaluates on the held-out test set -> these are your real, unbiased metrics

When training finishes, your model weights are saved to `models/<exp_name>/weights/best.pt`.

## Step 3 - Inspect training results (curves + metrics)

After training, look at the results:

**Metrics to check (from Phase 4 output):**

- **Precision** -> Of everything the model flagged as a fish, how many were real? (closer to 1.0 = fewer false alarms)
- **Recall** -> Of all real fish in the images, how many did the model find? (closer to 1.0 = fewer misses)
- **mAP@50** -> The standard summary score for detection quality. Higher is better
- **mAP@50-95** -> A stricter version that also requires precise bounding boxes

**Training curves** (in `models/<exp_name>/results.png`):

- Loss should generally decrease over epochs
- If training loss keeps dropping but validation metrics stop improving, the model may be overfitting. This is a common sign you need more data or a smaller model

> **Note:** With only 100 images and 20 epochs, don't expect perfect results! This is a demo; the goal is to see the full pipeline working.


## Step 4 - Run inference on a images with alien species

Now let's see what your model can actually detect on images it has **never seen**.

Open the notebook: [**Inference_playday.ipynb**](Inference_playday.ipynb).

**What you'll edit in Phase 1 (Configuration):**

- `model_path`: point to your trained model: `"models/<exp_name>/weights/best.pt"`
- `images_dir`: point to `"02_inference/images"`

Run the cells to see detection overlays on each image. Can your model spot the invasive *Neogobius melanostomus*?

**Try both models!** Go back to Phase 1 and swap `model_path`:

- Your model: `"models/<exp_name>/weights/best.pt"` (trained on 100 images with nano)
- Golden model: `"03_golden_model/weights/best.pt"` (trained on ~500 images with medium)

How do the detections compare? The golden model should be noticeably more confident and accurate -> that's the difference more data and a larger model make.

## Step 5 - Inspect the results

Look at the detection overlays from both models:

- Which species does each model detect?
- How confident are the detections? (shown as percentages on each bounding box)
- Does the golden model catch fish that your nano model missed?
- Try lowering `conf_thres` to `0.3` in the inference notebook -> do more (noisier) detections appear?

## Micro-glossary

| Term | What it means | Why it matters |
|:-----|:-------------|:---------------|
| **Epoch** | One complete pass over the training dataset. 20 epochs = the model sees each image 20 times. | More epochs give the model more chances to learn, but too many can lead to overfitting. |
| **Loss** | A number measuring how wrong the model's predictions are. Plotted as curves over training. | Should decrease over time. If training loss drops but validation loss rises, the model is memorising instead of learning. |
| **Precision** | Of everything the model predicted as a detection, how many were actually correct? | High precision = few false alarms. 0.80 means 80% of detections are real. |
| **Recall** | Of all real objects in the images, how many did the model find? | High recall = few misses. 0.80 means the model catches 80% of objects. |
| **mAP** | Mean Average Precision: a single summary score of detection quality across confidence thresholds. | The standard metric for comparing object-detection models. Reported as mAP@50 (lenient) and mAP@50-95 (strict). |
| **Overfitting** | When the model learns training images too specifically and fails to generalise to new data. | Common with small datasets and large models. This is why we use nano for ~100 images. |
| **Confidence threshold** | The minimum score a detection needs to be shown. | 0.5 = only show detections the model is ≥50% sure about. Lower catches more but adds noise. |
| **Batch size** | How many images the model processes at once during training. | Larger batches use more GPU memory. Reduce if you get out-of-memory errors. |
| **YOLO** | "You Only Look Once": a family of fast object-detection models. | Designed for real-time detection. We use YOLO11, one of the latest versions from Ultralytics.
| **data.yaml** | The configuration file that tells YOLO where your images are and what classes to detect. | Every YOLO dataset needs one. It lists paths to train/val/test splits and the class names. |


## Copying files to personal storage on EDITO

All files in this Jupyter instance will be deleted once you shut down your SUBSIM service.

To store files, you can run the following cell to copy file `PATH_TO_FILE` (edit this path) to your personal storage on EDITO.

Please see [EDITO documentation](https://docs.dive.edito.eu/articles/create/interact-with-personal-storage.html) for further instructions.

In [ ]:
!mc cp PATH_TO_FILE s3/$S3_BUCKET

After this command succeeds, you should see the copied file in [EDITO File Explorer](https://datalab.dive.edito.eu/file-explorer).